# Preprocessing and Dimensionality Reduction with PCAGlobal Geological Risk Analysis: complete preprocessing pipeline, PCA, and geospatial visualization.

In [1]:
import sys, os, warningswarnings.filterwarnings('ignore')project_root = os.path.join(os.getcwd(), '..')sys.path.insert(0, os.path.abspath(project_root))import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom src.data_loader import load_combined_datafrom src.preprocessing import preprocessing_pca_pipeline, load_pipeline, detect_skew_columnsfrom src.visualization import (    plot_cumulative_variance, plot_pca_2d, plot_biplot, plot_pca_interactive,    plot_risk_map, plot_pairplot_pca, plot_loadings_heatmap, plot_pca_3d,)sns.set_theme(style='whitegrid', context='paper', font='sans-serif', font_scale=1.2, palette='mako')plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})os.makedirs('figures', exist_ok=True)print('Libraries cargadas correctamente')

Libraries cargadas correctamente


## 1. Geological Data IngestionLoading combined data from multiple geological sources. The data is synthetic for pipeline development; the final ingestion will integrate H3, USGS, IBTrACS and GVP.

In [1]:
df_raw = load_combined_data()print(f'Shape: {df_raw.shape}')print(f'Columns: {list(df_raw.columns)}')print(f'Lat range: [{df_raw['lat'].min():.1f}, {df_raw['lat'].max():.1f}]')print(f'Lon range: [{df_raw['lon'].min():.1f}, {df_raw['lon'].max():.1f}]')df_raw.head()

Shape: (500, 10)
Columns: ['magnitud_max_sismo', 'profundidad_media_sismo', 'frecuencia_eventos_sismicos', 'viento_max_ciclones', 'presion_min_ciclones', 'elevacion_volcan', 'categoria_tormenta', 'tipo_volcan', 'lat', 'lon']
Lat range: [-60.0, 59.9]
Lon range: [-179.7, 179.1]


   magnitud_max_sismo  profundidad_media_sismo  ...        lat         lon
0            3.938536                 9.712413  ...  -0.292070 -160.143627
1            9.020243                33.146485  ... -41.418878  115.430962
2            5.633491                15.805201  ... -38.049249 -166.428939
3            4.825885                 4.657492  ... -16.194292  -47.818935
4            3.339250                 3.601204  ...  -8.107540   15.227991

[5 rows x 10 columns]

## 2. Exploratory Analysis of Numerical FeaturesDescriptive statistics and skewness detection on geological variables.

In [1]:
num_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()excluir = {'lat', 'lon', 'h3_index', 'cell_id'}features = [c for c in num_cols if c not in excluir]df_stats = df_raw[features].describe().Tdf_stats['skewness'] = df_raw[features].skew()df_stats['skew_alto'] = df_stats['skewness'].abs() > 0.75df_stats[['mean', 'std', 'skewness', 'skew_alto']]

                                   mean         std  skewness  skew_alto
magnitud_max_sismo             5.008618    1.948042  1.445241       True
profundidad_media_sismo       10.370757    9.949388  2.814861       True
frecuencia_eventos_sismicos    4.984000    2.334662  0.357108      False
viento_max_ciclones           61.291120   17.929350  1.024144       True
presion_min_ciclones         979.910418   24.677263  0.128157      False
elevacion_volcan             459.516607  731.358171  5.515789       True

## 3. Logarithmic Transformation (log1p)Apply log1p to high-skew columns to reduce asymmetry and stabilize variance before scaling.

In [1]:
skew_cols = detect_skew_columns(df_raw, umbral=0.75)print(f'Columns con skew alto (>0.75): {skew_cols}')df_log = df_raw.copy()for col in skew_cols:    df_log[f'{col}_log'] = np.log1p(df_log[col].clip(lower=0))skew_after = df_log[[f'{c}_log' for c in skew_cols if f'{c}_log' in df_log.columns]].skew()print('\nSkew after log1p:')print(skew_after.round(3))

TypeError: detect_skew_columns() got an unexpected keyword argument 'umbral'

## 4. One-Hot Encoding and StandardScalerEncode categorical variables and standardize (mean=0, variance=1) to ensure PCA is not biased by scale differences.

In [1]:
cat_cols = ['categoria_tormenta', 'tipo_volcan']dummy_cols = [c for c in cat_cols if c in df_log.columns]df_encoded = pd.get_dummies(df_log, columns=dummy_cols, drop_first=True)exclude_final = excluir | {'volcano_name', 'region', 'country'}cols_numeric = [c for c in df_encoded.select_dtypes(include=[np.number]).columns if c not in exclude_final]from sklearn.preprocessing import StandardScalerscaler = StandardScaler()X_scaled = scaler.fit_transform(df_encoded[cols_numeric])df_scaled = pd.DataFrame(X_scaled, columns=cols_numeric, index=df_raw.index)print(f'Scaled shape: {df_scaled.shape}')pd.DataFrame({'media': df_scaled.mean().round(6), 'std': df_scaled.std().round(6)}).head()

NameError: name 'df_log' is not defined

## 5. Dimensionality Reduction with PCAApply Principal Component Analysis to capture maximum variance in a few orthogonal components.

In [1]:
from sklearn.decomposition import PCApca = PCA(random_state=42)X_pca = pca.fit_transform(X_scaled)explained_var = pca.explained_variance_ratio_cum_var = np.cumsum(explained_var)n_85 = int(np.searchsorted(cum_var, 0.85) + 1)print(f'Variance explained by PC1: {explained_var[0]*100:.1f}%')print(f'Variance explained by PC2: {explained_var[1]*100:.1f}%')print(f'Components for 85%% variance: {n_85}')print(f'Cumulative variance with {n_85} comps: {cum_var[n_85-1]*100:.1f}%')df_pca = pd.DataFrame(    X_pca[:, :n_85],    columns=[f'PC{i+1}' for i in range(n_85)],    index=df_raw.index)df_pca.head()

NameError: name 'X_scaled' is not defined

### 5.1 Cumulative Explained Variance

In [1]:
plot_cumulative_variance(pca, threshold=0.85, save_path='figures/variance_acumulada.png')plt.show()

AttributeError: 'PCA' object has no attribute 'explained_variance_ratio_'

### 5.2 2D Projection

In [1]:
plot_pca_2d(df_pca, pca_model=pca, save_path='figures/pca_2d.png')plt.show()

NameError: name 'df_pca' is not defined

### 5.3 Biplot: Original Variable Contributions

In [1]:
plot_biplot(df_pca, pca_model=pca, feature_names=cols_numeric, save_path='figures/biplot.png')plt.show()

NameError: name 'df_pca' is not defined

### 5.4 Interactive PCA 3D (Plotly)

In [1]:
plot_pca_3d(df_pca, pca_model=pca, save_path='figures/pca_3d.html')print('3D plot saved to figures/pca_3d.html')

NameError: name 'df_pca' is not defined

## 6. Loadings: Which Variables Contribute to Each Component?The heatmap shows which geological variables (seismicity, cyclones, volcanoes) have the highest weight in each principal component.

In [1]:
plot_loadings_heatmap(pca, feature_names=cols_numeric, n_components=n_85, save_path='figures/loadings_heatmap.png')plt.show()

NameError: name 'cols_numeric' is not defined

## 7. Principal Components Pairplot

In [1]:
plot_pairplot_pca(df_pca, n_components=min(4, n_85), save_path='figures/pairplot_pca.png')plt.show()

NameError: name 'df_pca' is not defined

## 8. Geographical Risk Map (Folium)Each point represents a geographical cell. Color indicates PC1 value (higher = more extreme geological signal). Popups show seismic magnitude, cyclone wind, and volcano elevation.

In [1]:
df_mapa = df_raw[['lat', 'lon', *features]].copy()df_mapa['PC1'] = df_pca['PC1'].valuesplot_risk_map(    df_mapa,    lat_col='lat', lon_col='lon',    color_col='PC1',    popup_cols=['magnitud_max_sismo', 'viento_max_ciclones', 'elevacion_volcan'],    radius_scale=2.0,    save_path='figures/mapa_riesgo.html')print('Interactive map saved to figures/mapa_riesgo.html')

NameError: name 'df_pca' is not defined

## 9. Interactive 2D Plot (Plotly)

In [1]:
plot_pca_interactive(df_pca, pca_model=pca, save_path='figures/pca_interactivo.html')print('HTML saved to figures/pca_interactivo.html')

NameError: name 'df_pca' is not defined

## 10. Production-Ready Pipeline ExportBuild an sklearn Pipeline with SkewLogTransformer, OneHotTransformer, StandardScaler and PCA. Export with joblib for production reuse.

In [1]:
df_pipe, pipeline, _ = preprocessing_pca_pipeline(    df_raw, target_variance=0.85, save_path='models/pipeline_riesgo.joblib')print(f'Pipeline exported: {df_pipe.shape[1]} components')print(f'Explained variance: {pipeline.named_steps['pca'].explained_variance_ratio_.cumsum()[-1]:.2%}')loaded = load_pipeline('models/pipeline_riesgo.joblib')X_test = loaded.transform(df_raw.head(5))print(f'Transform on 5 muestras: {X_test.shape}')

Pipeline exported: 5 components
Explained variance: 86.42%
Transform on 5 muestras: (5, 5)


## 11. Save Processed Data

In [1]:
df_pca.to_csv('data/processed/dataset_pca.csv', index=False)df_scaled.to_csv('data/processed/dataset_scaled.csv', index=False)print('Data saved to data/processed/')

NameError: name 'df_pca' is not defined